## Импорты

In [1]:
CONFIG_NAME = "wav2vec2.yaml"

In [2]:
import sys
import os
from pathlib import Path

# Допустим, что ноутбук находится в той же директории, что и папка acoustic/
sys.path.insert(0, str(Path.cwd()))

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("transformers.generation_utils").setLevel(logging.ERROR)

In [3]:
from acoustic.utils.config import load_config
from acoustic.dataset.load_dataset import load_and_prepare_dataset
from acoustic.models.load_model import build_model
from acoustic.training.load_metrics import load_metrics
from acoustic.training.callbacks import get_callback
from acoustic.training import get_trainer_class


## Загрузка конфигурации

In [4]:
CONFIG_PATH = f"acoustic/configs/{CONFIG_NAME}"

cfg = load_config(CONFIG_PATH, overrides=None)

print("Configuration loaded")

Configuration loaded


## Загрузка датасета

In [5]:
dataset = load_and_prepare_dataset(cfg)

## Инициализация модели

In [6]:
model, processor, data_collator = build_model(cfg)

print("Model built")

Some weights of the model checkpoint at jonatasgrosman/wav2vec2-large-xlsr-53-russian were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at jonatasgrosman/wav2vec2-large-xlsr-53-russian and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
Y

Model built


## Создание и загрузка метрик, callbacks, trainer

In [7]:
metrics_list = load_metrics(cfg['training']['metrics'])
print(f"Metrics: {cfg['training']['metrics']}")

callbacks = []
for cb_name in cfg['training']['callbacks']:
    callbacks.append(get_callback(cb_name))

Metrics: ['wer', 'cer', 'f1', 'detailed_stats', 'ser', 'space_wer']


In [8]:

trainer_name = cfg['training'].get('trainer', 'BaseTrainer')
TrainerClass = get_trainer_class(trainer_name)

trainer = TrainerClass(
    cfg=cfg,
    model=model,
    processor=processor,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    metrics=metrics_list,
    callbacks=callbacks,
    data_collator=data_collator
)

## Обучение

In [9]:
print("Starting training")
trainer.train()

Starting training


Training: 100%|██████████| 800/800 [00:00<?, ?step/s]

{'train_runtime': 0.6318, 'train_samples_per_second': 20266.796, 'train_steps_per_second': 1266.18, 'train_loss': 0.0, 'epoch': 5.0}


## Проверка

In [11]:
eval_dataset = dataset['validation']

print("Demo on validation examples")
import random
import torch
from acoustic.models import get_generate_method

builder_key = cfg['model']['builder']
generate_fn = get_generate_method(builder_key)

if eval_dataset and len(eval_dataset) > 0:
    indices = random.sample(range(len(eval_dataset)), min(3, len(eval_dataset)))
    device = next(model.parameters()).device
    model.eval()
    
    for i in indices:
        example = eval_dataset[i]
        audio_array = example["audio"]["array"]
        ref_text = example["sentence"]
        
        inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt")
        input_key = "input_features" if "input_features" in inputs else "input_values"
        input_data = inputs[input_key].to(device)
        
        with torch.no_grad():
            predicted_ids = generate_fn(model, input_data, processor)
            pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        
        print(f"\nExample {i+1}:")
        print(f" Reference: {ref_text}")
        print(f" Prediction: {pred_text}")
else:
    print("No validation dataset for demo.")

Demo on validation examples

Example 141:
 Reference: баба шьям были поданы 108 тарелок чаппан-бхог в индуизме 56 различных яств включая сладости фрукты орехи и другие блюда преподносимые божеству
 Prediction: бабашьям были поданы чопанбх в индуизме различных яств включая сладости фрукты арехи и другие блюда приподносимые божеству

Example 126:
 Reference: это пятый cep мартейи за четыре года
 Prediction: это пятый сер мартейи за четыр года

Example 115:
 Reference: учитывая то что в день можно было выиграть только восемнадцать медалей ряду стран не удалось оказаться на пьедестале почёта
 Prediction: учитывая то что в день можно было выиграть только цать медалей ряду стран не удалось оказаться на пьедестале почёта
